In [48]:
import cv2 
import numpy as np
import csv

In [ ]:
image = "images/20260202_pdau4.jpg"
img = cv2.imread(image)
image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(image, 200, 255, cv2.THRESH_BINARY)
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

In [50]:
points = []
scalebar_pixel = 0
scalebar_length = input("Measurement of Scalebar(no units): ")
for i, cnt  in enumerate(contours):
  area = cv2.contourArea(cnt)
  x, y, w, h = cv2.boundingRect(cnt)
  
  if area > 200:
    x, y, w, h = cv2.boundingRect(cnt)
    yn, ym = y - 2, y + h + 2
    xn, xm = x - 2, x + w + 2
    roi = image[yn:ym, xn:xm]
    _, bin = cv2.threshold(roi, 200, 255, cv2.THRESH_BINARY)
    roi_edge = cv2.Canny(bin, threshold1=50, threshold2=150)
    lines = cv2.HoughLinesP(roi_edge, 1, np.pi/180, threshold=200, minLineLength=200, maxLineGap=1)
    if lines is not None:
      pixel_row = np.sum(bin, axis = 1)
      y_pos = np.argmax(pixel_row)
      column = np.where(bin[y_pos, :] == 255)[0]
      xe = column[-1]
      xs = column[0]
      scalebar_pixel = xe - xs + 1
  rect = cv2.minAreaRect(cnt)
  box = cv2.boxPoints(rect)
  box = np.int64(box)
  points.append(box)

def Conversion(pixels, scalebar_length, scalebar_pixel):
  pixels = np.asarray(pixels).astype(float)
  scalebar_length = np.asarray(scalebar_length).astype(float)
  scalebar_pixel = np.asarray(scalebar_pixel).astype(float)
  measurement = (pixels * scalebar_length) / scalebar_pixel
  if measurement.ndim == 0:
    return float(measurement)
  return measurement 

In [51]:
shapes_array = np.array(points)
for i, cnt  in enumerate(contours):
  nearby_corners_indexes = []
  for target_point in points[i]:
    search_radius = 100
    corner_distances = np.linalg.norm(shapes_array - target_point, axis=2)
    inside_mask = corner_distances <= search_radius
    matching_indices = np.any(inside_mask, axis=1)
    original_indices = np.flatnonzero(matching_indices)
    inside_shapes = shapes_array[original_indices]
    for nearby in original_indices:
      nearby_corners_indexes.append(nearby)
  nearby_corners_indexes = np.unique(nearby_corners_indexes)
  nearby_corners_indexes = nearby_corners_indexes[nearby_corners_indexes != i]
  for nearby_contours in nearby_corners_indexes:
    pts1 = points[nearby_contours]
    pts2 = points[i]
    success, intersection_pts = cv2.intersectConvexConvex(pts1, pts2)
    if success and intersection_pts is not None and len(intersection_pts) > 0:
      intersection_pts = np.int32(intersection_pts)
      x, y, w, h = cv2.boundingRect(intersection_pts)
      top_left = (x, y)
      bottom_right = (x + w, y + h)
      cv2.rectangle(img, top_left, bottom_right, (0, 0, 255), 2)

In [52]:
a = 0
measurements = []
for i, cnt  in enumerate(contours):
  area = cv2.contourArea(cnt)
  rect = cv2.minAreaRect(cnt)
  box = cv2.boxPoints(rect)
  box = np.int64(box)
  side1 = np.linalg.norm(box[0] - box[1])
  side2 = np.linalg.norm(box[1] - box[2])
  length = max(side1, side2)
  width = min(side1, side2)
  length = np.int64(length)
  width = np.int64(width)
  x, y, w, h = cv2.boundingRect(cnt)
  cX = w// 2 + x
  cY = h// 2 + y
  length = Conversion(length, scalebar_length, scalebar_pixel)
  width = Conversion(width, scalebar_length, scalebar_pixel)
  area = Conversion(area, scalebar_length, scalebar_pixel)
  measurements.append([i, length, width, area, cX, cY])
  cv2.putText(
  img, 
  text=f"#{i}", 
  org=(cX - 10, cY + 5), 
  fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
  fontScale=0.5, 
  color=(255, 0, 0), 
  thickness=2
)

In [ ]:
ideal_flakes = []
for contour in measurements:
    if (contour[1] > 50 and contour[2] > 10) or ((abs(contour[1] - contour[2]) <= 20) and (contour[1] > 30 and contour[2] > 30)):
        cv2.drawContours(img, [points[contour[0]]], 0, (0,255,0), 2)
        ideal_flakes.append(contour)
cv2.imwrite('flakes_highlighted.jpg', img) 

True

In [ ]:
with open('flake_table.csv', 'w', newline='') as file:
    writer = csv.writer(file)

    field = ['#-marker', 'length', 'width', 'area', 'x-cord', 'y-cord']
    writer.writerow(field)
    for ideal in ideal_flakes:
        writer.writerow(ideal)
